In [7]:
import cadquery as cq

# Trapezoid prism dimensions
bottom_width = 150
top_width = 150
height = 150
depth = 150

trapezoid_points = [
    (0, 0),
    (0, height),
    (top_width, 50),
    (bottom_width, 0),
]

result = (
    cq.Workplane("ZY")
    .polyline(trapezoid_points)
    .close()
    .extrude(-depth)
)

# Model-space axes, useful for checking direction after restarting the kernel.
axis_len = 20
axis_radius = 0.2
x_axis = cq.Workplane("YZ").circle(axis_radius).extrude(axis_len)
y_axis = cq.Workplane("ZX").circle(axis_radius).extrude(axis_len)
z_axis = cq.Workplane("XY").circle(axis_radius).extrude(axis_len)

assy = cq.Assembly(name="trapezoid_with_axes")
assy.add(result, name="trapezoid", color=cq.Color("gold"))
assy.add(x_axis, name="x_axis", color=cq.Color("red"))
assy.add(y_axis, name="y_axis", color=cq.Color("green"))
assy.add(z_axis, name="z_axis", color=cq.Color("blue"))

assy


In [8]:
step_path = r"C:\Users\j1876\Documents\Cadquery\cadquery\revo-scanned\0430_01_mesh_auto1.stp"

# Calibration controls: edit these values, then re-run this cell.
rotate_x = -60
rotate_y = 0
rotate_z = 10
manual_move = (85, 100, 95)

scan_step = cq.importers.importStep(step_path)

# Move the imported STEP bounding-box center to the global origin first.
bb = scan_step.val().BoundingBox()
origin_offset = (-bb.center.x, -bb.center.y, -bb.center.z)
scan_step = scan_step.translate(origin_offset)

# Then rotate around the global origin, in degrees.
scan_step = scan_step.rotate((0, 0, 0), (1, 0, 0), rotate_x)
scan_step = scan_step.rotate((0, 0, 0), (0, 1, 0), rotate_y)
scan_step = scan_step.rotate((0, 0, 0), (0, 0, 1), rotate_z)

# Finally apply a manual translation for fine alignment.
scan_step = scan_step.translate(manual_move)

# Re-running this cell should replace the imported STEP part, not duplicate it.
assy.children = [child for child in assy.children if child.name != "scan_step"]
assy.add(scan_step, name="scan_step", color=cq.Color(0.75, 0.75, 0.75, 0.55))

origin_offset
assy


In [9]:
# Use the largest closed solid inside the scanned STEP as the cup cutter.
# Do not move or project the cup: this removes only the actual overlap between
# the holder and the placed cup solid.
from OCP.TopAbs import TopAbs_SOLID
from OCP.TopExp import TopExp_Explorer


def largest_solid_from_compound(wp):
    """Return the largest solid contained in an imported STEP compound."""
    solid_exp = TopExp_Explorer(wp.val().wrapped, TopAbs_SOLID)
    largest = None
    largest_volume = -1

    while solid_exp.More():
        solid = cq.Solid(solid_exp.Current())
        volume = abs(solid.Volume())
        if volume > largest_volume:
            largest = solid
            largest_volume = volume
        solid_exp.Next()

    if largest is None:
        raise RuntimeError("No solid found in scan_step; cannot run a volume boolean cut.")
    return cq.Workplane(obj=largest), largest_volume


cup_cutter, cup_cutter_volume = largest_solid_from_compound(scan_step)
holder_with_pocket = result.cut(cup_cutter).clean()
removed_volume = result.val().Volume() - holder_with_pocket.val().Volume()
print({"cup_cutter_volume": cup_cutter_volume, "removed_volume": removed_volume})

# Re-running this cell should replace the carved holder, not stack copies.
assy.children = [
    child for child in assy.children
    if child.name not in (
        "trapezoid", "scan_step", "scan_auto_fit", "cup_cutter", "holder_with_pocket",
    )
]
assy.add(holder_with_pocket, name="holder_with_pocket", color=cq.Color("gold"))
# Uncomment this line if you want to see the transparent largest-solid cutter.
# assy.add(cup_cutter, name="cup_cutter", color=cq.Color(1, 0, 0, 0.25))

assy


{'cup_cutter_volume': 335159.27423567954, 'removed_volume': 133257.44317746768}


In [ ]:
# Export the carved holder for 3D printing.
from pathlib import Path

out_dir = Path("exports")
out_dir.mkdir(exist_ok=True)

stl_path = out_dir / "holder_with_cup_pocket.stl"
step_path = out_dir / "holder_with_cup_pocket.step"

# STL is for slicing/3D printing; STEP is a reusable CAD backup.
cq.exporters.export(holder_with_pocket, str(stl_path), tolerance=0.1, angularTolerance=0.1)
cq.exporters.export(holder_with_pocket, str(step_path))

print(f"STL exported to: {stl_path.resolve()}")
print(f"STEP exported to: {step_path.resolve()}")


In [ ]:
# Experimental auto-fit cell: coarse search + bounding-box placement.
# This is a fast initial alignment, not a full collision/physics solver.

# Controls
show_auto_fit_preview = False
allow_auto_scale = False
clearance = 2.0
fit_margin = 0.85

# Candidate rotations in degrees. Add finer values after the coarse result looks close.
candidate_x = range(-90, 16, 15)
candidate_y = [0]
candidate_z = range(0, 360, 30)

def bbox_info(shape):
    bb = shape.BoundingBox()
    return {
        "bb": bb,
        "center": (bb.center.x, bb.center.y, bb.center.z),
        "size": (bb.xlen, bb.ylen, bb.zlen),
    }

def center_shape_at_origin(shape):
    bb = shape.BoundingBox()
    return shape.translate((-bb.center.x, -bb.center.y, -bb.center.z))

def rotated_shape(shape, rx, ry, rz):
    rotated = shape.rotate((0, 0, 0), (1, 0, 0), rx)
    rotated = rotated.rotate((0, 0, 0), (0, 1, 0), ry)
    rotated = rotated.rotate((0, 0, 0), (0, 0, 1), rz)
    return rotated

def scaled_about_origin(shape, scale):
    return shape.scale(scale) if scale != 1 else shape

# Holder bounds from the trapezoid assembly object.
holder_bb = result.val().BoundingBox()
holder_x_span = holder_bb.xlen * fit_margin
holder_y_span = holder_bb.ylen * fit_margin
holder_center_x = holder_bb.center.x
holder_center_y = holder_bb.center.y

# Approximate sloped top surface from the trapezoid sketch.
# In this model, the upper edge goes from local (0, height) to (top_width, 50).
def shelf_top_z_at_y(y):
    if bottom_width == 0:
        return height
    t = max(0, min(1, y / bottom_width))
    return height + (50 - height) * t

target_center = (
    holder_center_x,
    holder_center_y,
    shelf_top_z_at_y(holder_center_y),
)

# Import and normalize the incoming object.
raw_scan = cq.importers.importStep(step_path).val()
centered_scan = center_shape_at_origin(raw_scan)

best = None
for rx in candidate_x:
    for ry in candidate_y:
        for rz in candidate_z:
            trial = rotated_shape(centered_scan, rx, ry, rz)
            info = bbox_info(trial)
            sx, sy, sz = info["size"]
            required_scale = min(holder_x_span / sx, holder_y_span / sy) if sx and sy else 0
            scale = min(1, required_scale) if allow_auto_scale else 1

            fits = sx * scale <= holder_x_span and sy * scale <= holder_y_span
            overflow = max(0, sx * scale - holder_x_span) + max(0, sy * scale - holder_y_span)
            shrink_penalty = abs(1 - scale) * 1000
            height_penalty = sz * scale * 0.01
            score = overflow * 100 + shrink_penalty + height_penalty

            if best is None or score < best["score"]:
                best = {
                    "score": score,
                    "fits": fits,
                    "rotation": (rx, ry, rz),
                    "scale": scale,
                    "bbox_size_before_place": info["size"],
                    "shape": scaled_about_origin(trial, scale),
                }

auto_scan = best["shape"]
placed_bb = auto_scan.BoundingBox()
place_move = (
    target_center[0] - placed_bb.center.x,
    target_center[1] - placed_bb.center.y,
    target_center[2] + clearance - placed_bb.zmin,
)
auto_scan = auto_scan.translate(place_move)

auto_fit_report = {
    "rotation_xyz_deg": best["rotation"],
    "scale": best["scale"],
    "fits_bbox_footprint": best["fits"],
    "bbox_size_before_place": best["bbox_size_before_place"],
    "place_move": place_move,
    "target_center": target_center,
}

# Replace old scan objects so repeated runs do not stack copies.
assy.children = [
    child for child in assy.children
    if child.name not in ("scan_step", "scan_auto_fit")
]
assy.add(auto_scan, name="scan_auto_fit", color=cq.Color(0.1, 0.65, 1.0, 0.55))

print(auto_fit_report)
if show_auto_fit_preview:
    assy
else:
    'Set show_auto_fit_preview = True to render the 3D assembly.'


In [ ]:
assy